# 🌊 Odysseus Streaming Analytics: Athena, DuckDB & Parquet S3 Tier

**Proyecto:** Odysseus AI Platform & Predictive MLOps Framework  
**Lead Architect:** [Guillen Concepcion](https://www.linkedin.com/in/guillen-concepcion-25266b127) *(Senior Data Scientist & MLOps Engineer)*  
**Componente:** Ad-Hoc Serverless SQL & Temporal Analysis of Curated Sensor Telemetry  

---

## 🧭 Overview
This notebook demonstrates downstream analytics over the transactional columnar Parquet partitions rolled by Apache Flink's `FileSink` into Amazon S3 (`s3://.../curated/year=YYYY/month=MM/day=DD/hour=HH/`).

Using **DuckDB** (as an in-process vectorized engine representing Amazon Athena query execution), we validate:
1. **Partition Pruning:** Sub-second queries targeting virtual partition keys without AWS Glue Crawler lag.
2. **Distributional Statistics:** Reporting medians and 95th percentiles (P95) to respect Odysseus statistical rigor on skewed industrial metrics.
3. **Thermal-Kinetic Fault Correlation:** Visualizing the multi-stage anomaly regime detected by the Flink CEP engine.

In [ ]:
import os
import numpy as np
import pandas as pd
import duckdb

# Ensure consistent formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

print(f"DuckDB Version: {duckdb.__version__}")
print(f"Pandas Version: {pd.__version__}")

## 1. Synthetic Parquet Dataset Generator (Matching Flink FileSink Output)
In a live deployment, these Parquet files are written directly by Flink TaskManagers. For standalone execution of this notebook, we simulate the exact schema committed by `StreamingJob.java`.

In [ ]:
np.random.seed(42)
n_samples = 5000
sensors = [f"TURBINE-{i:03d}" for i in range(1, 21)]

data = []
base_time = 1794396000000  # Base Epoch Millis

for i in range(n_samples):
    s_id = np.random.choice(sensors)
    t_start = base_time + (i * 10000)  # 10s sliding steps
    t_end = t_start + 60000            # 1 min window
    
    # Standard operating regime
    avg_temp = float(np.random.normal(58.5, 3.2))
    max_temp = float(avg_temp + np.random.exponential(2.5))
    avg_vib = float(np.random.normal(1.85, 0.35))
    max_vib = float(avg_vib + np.random.exponential(0.6))
    avg_press = float(np.random.normal(104.2, 1.1))
    avg_power = float(np.random.normal(44.0, 4.5))
    min_batt = float(24.0 - np.random.uniform(0.1, 0.4))
    
    # Injected CEP anomaly signature in 2% of windows
    is_anomaly = False
    if np.random.rand() < 0.025:
        max_temp = float(np.random.uniform(86.5, 95.0))
        max_vib = float(np.random.uniform(4.8, 6.4))
        avg_power += 12.0
        is_anomaly = True
        
    data.append({
        "sensor_id": s_id,
        "window_start": t_start,
        "window_end": t_end,
        "avg_temperature": avg_temp,
        "max_temperature": max_temp,
        "avg_vibration": avg_vib,
        "max_vibration": max_vib,
        "avg_pressure": avg_press,
        "avg_power": avg_power,
        "min_battery": min_batt,
        "sample_count": np.random.randint(55, 65),
        "anomaly_flag": is_anomaly,
        "year": "2026",
        "month": "11",
        "day": "11",
        "hour": "08"
    })

df_telemetry = pd.DataFrame(data)
os.makedirs("../data/curated/year=2026/month=11/day=11/hour=08", exist_ok=True)
parquet_path = "../data/curated/year=2026/month=11/day=11/hour=08/part-0.parquet"
df_telemetry.to_parquet(parquet_path, compression="snappy", index=False)
print(f"[+] Wrote {len(df_telemetry):,} rows to {parquet_path} ({os.path.getsize(parquet_path) / 1024:.1f} KB)")

## 2. Serverless SQL Queries (Amazon Athena / DuckDB Emulation)
We query the Parquet files using ANSI SQL with zero crawling overhead.

In [ ]:
con = duckdb.connect(database=':memory:')
con.execute(f"CREATE VIEW curated_sensor_metrics AS SELECT * FROM read_parquet('{parquet_path}')")

# Query 1: Top 10 High-Severity Anomalies detected by Flink
query_anomalies = """
SELECT 
    sensor_id,
    epoch_ms(window_start) AS window_start_time,
    round(max_temperature, 2) AS max_temp_c,
    round(max_vibration, 3)   AS max_vib_rms,
    round(avg_power, 2)       AS avg_power_kw,
    sample_count
FROM curated_sensor_metrics
WHERE anomaly_flag = true
ORDER BY max_vibration DESC
LIMIT 10;
"""

df_top_anomalies = con.execute(query_anomalies).fetchdf()
df_top_anomalies

## 3. Odysseus Statistical Rigor: Distributional Percentiles across Assets
Per **Odysseus Rule 1 (Statistical Rigor over Mechanical Arithmetic)**: In skewed industrial telemetry distributions, we never rely solely on arithmetic means. We evaluate median (P50), 95th percentile (P95), and the Interquartile Range (IQR).

In [ ]:
query_percentiles = """
SELECT 
    sensor_id,
    count(*) AS total_windows,
    sum(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS anomaly_windows,
    round(quantile_cont(avg_temperature, 0.50), 2) AS median_temp,
    round(quantile_cont(max_temperature, 0.95), 2) AS p95_max_temp,
    round(quantile_cont(avg_vibration, 0.50), 3)   AS median_vib,
    round(quantile_cont(max_vibration, 0.95), 3)   AS p95_max_vib
FROM curated_sensor_metrics
GROUP BY sensor_id
ORDER BY anomaly_windows DESC, p95_max_vib DESC
LIMIT 10;
"""

df_fleet_stats = con.execute(query_percentiles).fetchdf()
df_fleet_stats

## 4. Visual Diagnostics: Multi-Metric Anomaly Separation
Plotting the thermal vs vibration phase space demonstrates the distinct cluster isolated by the Flink CEP state machine.

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    
    normal = df_telemetry[~df_telemetry["anomaly_flag"]]
    anomalies = df_telemetry[df_telemetry["anomaly_flag"]]
    
    ax.scatter(normal["max_temperature"], normal["max_vibration"], 
               c="#38bdf8", alpha=0.35, s=25, label="Nominal Windows")
    ax.scatter(anomalies["max_temperature"], anomalies["max_vibration"], 
               c="#ef4444", alpha=0.9, s=60, edgecolors="black", label="Flink CEP Anomaly Flagged")
    
    # CEP Threshold Lines
    ax.axvline(85.0, color="#f59e0b", linestyle="--", linewidth=1.5, label="Thermal Threshold (85°C)")
    ax.axhline(4.5, color="#ec4899", linestyle="--", linewidth=1.5, label="Vibration Threshold (4.5 mm/s)")
    
    ax.set_title("Thermal vs Kinetic RMS Phase Space (Flink CEP Anomaly Separation)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Max Window Temperature (°C)", fontsize=10)
    ax.set_ylabel("Max Window RMS Vibration (mm/s)", fontsize=10)
    ax.legend(frameon=True, facecolor="#f8fafc", edgecolor="#cbd5e1", fontsize=9)
    ax.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is not installed in this environment. Run 'pip install matplotlib' to display plots.")